In [7]:
import re

str_ = """
                v_blk += numpy.einsum(
                    "jbkc,ia->ijkabc",
                    eris_OVOV[:, b0:b1, :, c0:c1],
                    t1a[:, a0:a1],
                )
                v_blk += numpy.einsum(
                    "iakc,jb->ijkabc",
                    eris_ovOV[:, a0:a1, :, c0:c1],
                    t1b[:, b0:b1],
                )
                v_blk += numpy.einsum(
                    "iakc,jb->ijkabc",
                    eris_ovOV[:, a0:a1, :, c0:c1],
                    t1b[:, b0:b1],
                )
                v_blk += (
                    numpy.einsum(
                        "JKBC,ai->iJKaBC",
                        t2bb[:, :, b0:b1, c0:c1],
                        fvo[a0:a1, :],
                    )
                    * 0.5
                )
                v_blk += (
                    numpy.einsum(
                        "iKaC,BJ->iJKaBC",
                        t2ab[:, :, a0:a1, c0:c1],
                        fVO[b0:b1, :],
                    )
                    * 2
                )
"""
lines = []
for line in str_.split("\n"):
    if "_blk" in line:
        lines.append("")
    if len(lines):
        lines[-1] = f"{lines[-1]}\n{line}"

translate_list = [
    {"j": "k", "k": "j", "-": "+", "+": "-"},
    {"j": "k", "k": "j", "c": "b", "b": "c"},
    {"c": "b", "b": "c", "-": "+", "+": "-"},
]
translate_list_extended = []
for dict_item in translate_list:
    dict_item_extended = dict_item.copy()
    for key, value in dict_item.items():
        dict_item_extended[key.upper()] = value.upper()
    translate_list_extended.append(dict_item_extended)
translate_list = translate_list_extended
print(translate_list)

print(str_)
print("Translated Results:")
for translate_list_item in translate_list:
    for line in lines:
        for subline in line.split("\n"):
            if_translated = False
            if "->" in subline:
                parts = subline.split("->")
                new_left = parts[0].translate(str.maketrans(translate_list_item))
                new_right = parts[1]
                newsubline = f"{new_left}->{new_right}"
                if_translated = True
                print(newsubline)

            # using re to capture the numpy index and do the replacement,
            # ie [:, :, b0:b1, c0:c1] and [:, a0:a1, :, :]
            pattern = r"\[(([a-z0-9:]+)[,\s]*)+\]"
            match = re.search(pattern, subline)
            if match:
                index_str = match.group(0)
                new_index_str = index_str.translate(str.maketrans(translate_list_item))
                newsubline = subline.replace(index_str, new_index_str)
                if_translated = True
                print(newsubline)

            pattern = r'[-+]='
            match = re.search(pattern, subline)
            if match:
                operator_str = match.group(0)
                new_operator_str = operator_str.translate(str.maketrans(translate_list_item))
                newsubline = subline.replace(operator_str, new_operator_str)
                if_translated = True
                print(newsubline)

            if not if_translated:
                if subline != "\n":
                    print(subline)

[{'j': 'k', 'k': 'j', '-': '+', '+': '-', 'J': 'K', 'K': 'J'}, {'j': 'k', 'k': 'j', 'c': 'b', 'b': 'c', 'J': 'K', 'K': 'J', 'C': 'B', 'B': 'C'}, {'c': 'b', 'b': 'c', '-': '+', '+': '-', 'C': 'B', 'B': 'C'}]

                v_blk += numpy.einsum(
                    "jbkc,ia->ijkabc",
                    eris_OVOV[:, b0:b1, :, c0:c1],
                    t1a[:, a0:a1],
                )
                v_blk += numpy.einsum(
                    "iakc,jb->ijkabc",
                    eris_ovOV[:, a0:a1, :, c0:c1],
                    t1b[:, b0:b1],
                )
                v_blk += numpy.einsum(
                    "iakc,jb->ijkabc",
                    eris_ovOV[:, a0:a1, :, c0:c1],
                    t1b[:, b0:b1],
                )
                v_blk += (
                    numpy.einsum(
                        "JKBC,ai->iJKaBC",
                        t2bb[:, :, b0:b1, c0:c1],
                        fvo[a0:a1, :],
                    )
                    * 0.5
     

In [17]:
import re

str_ = """
            v_blk = numpy.einsum(
                "jbkc,ia->ijkabc",
                eris_OVOV,
                t1a,
            )
            v_blk += numpy.einsum(
                "iakc,jb->ijkabc",
                eris_ovOV,
                t1b,
            )
            v_blk += numpy.einsum(
                "iakc,jb->ijkabc",
                eris_ovOV,
                t1b,
            )
            v_blk += (
                numpy.einsum(
                    "JKBC,ai->iJKaBC",
                    t2bb,
                    fvo,
                )
                * 0.5
            )
            v_blk += (
                numpy.einsum(
                    "iKaC,BJ->iJKaBC",
                    t2ab,
                    fVO,
                )
                * 2
            )

            w_blk -= (
                numpy.einsum(
                    "ikae,jceb->ijkabc",
                    t2ab,
                    eris_OVVV,
                )
                * 2
            )
            w_blk -= (
                numpy.einsum(
                    "ikeb,jcea->ijkabc",
                    t2ab,
                    eris_OVvv,
                )
                * 2
            )
            w_blk -= numpy.einsum(
                "kjbe,iaec->ijkabc",
                t2bb,
                eris_ovVV,
            )
            w_blk += (
                numpy.einsum(
                    "imab,jckm->ijkabc",
                    t2ab,
                    eris_OVOO,
                )
                * 2
            )
            w_blk += (
                numpy.einsum(
                    "mkab,jcim->ijkabc",
                    t2ab,
                    eris_OVoo,
                )
                * 2
            )
            w_blk += numpy.einsum(
                "kmbc,iajm->ijkabc",
                t2bb,
                eris_ovOO,
            )

            w_blk += (
                numpy.einsum(
                    "ikae,jbec->ijkabc",
                    t2ab,
                    eris_OVVV,
                )
                * 2
            )
            w_blk += (
                numpy.einsum(
                    "ikec,jbea->ijkabc",
                    t2ab,
                    eris_OVvv,
                )
                * 2
            )
            w_blk += numpy.einsum(
                "kjce,iaeb->ijkabc",
                t2bb,
                eris_ovVV,
            )
            w_blk -= (
                numpy.einsum(
                    "imac,jbkm->ijkabc",
                    t2ab,
                    eris_OVOO,
                )
                * 2
            )
            w_blk -= (
                numpy.einsum(
                    "mkac,jbim->ijkabc",
                    t2ab,
                    eris_OVoo,
                )
                * 2
            )
            w_blk -= numpy.einsum(
                "kmcb,iajm->ijkabc",
                t2bb,
                eris_ovOO,
            )

            w_blk -= (
                numpy.einsum(
                    "ijae,kbec->ijkabc",
                    t2ab,
                    eris_OVVV,
                )
                * 2
            )
            w_blk -= (
                numpy.einsum(
                    "ijec,kbea->ijkabc",
                    t2ab,
                    eris_OVvv,
                )
                * 2
            )
            w_blk -= numpy.einsum(
                "jkce,iaeb->ijkabc",
                t2bb,
                eris_ovVV,
            )
            w_blk += (
                numpy.einsum(
                    "imac,kbjm->ijkabc",
                    t2ab,
                    eris_OVOO,
                )
                * 2
            )
            w_blk += (
                numpy.einsum(
                    "mjac,kbim->ijkabc",
                    t2ab,
                    eris_OVoo,
                )
                * 2
            )
            w_blk += numpy.einsum(
                "jmcb,iakm->ijkabc",
                t2bb,
                eris_ovOO,
            )
"""
replace_char_dict = {"k": "k0:k1", "c": "c0:c1"}
lines = []
for line in str_.split("\n"):
    if "_blk" in line:
        lines.append("")
    if len(lines):
        lines[-1] = f"{lines[-1]}\n{line}"

for line in lines:
    index_chars_list = ["", ""]
    for subline in line.split("\n"):
        if len(index_chars_list) != 0:
            if index_chars_list[0] != "":
                subline = subline.replace(",", index_chars_list[0] + ",")
                index_chars_list.pop(0)

        pattern = r"[a-zA-Z,]+->[a-zA-Z,]+"
        match = re.search(pattern, subline)
        if match:
            index_chars_list[0] += "["
            index_chars_list[1] += "["
            index_str = match.group(0)
            parts = index_str.split("->")
            left_part, right_part = parts[0].split(",")
            for char in left_part.lower():
                if char in replace_char_dict:
                    index_chars_list[0] += replace_char_dict[char] + ","
                else:
                    index_chars_list[0] += ":,"
            for char in right_part.lower():
                if char in replace_char_dict:
                    index_chars_list[1] += replace_char_dict[char] + ","
                else:
                    index_chars_list[1] += ":,"
            index_chars_list[0] = index_chars_list[0][:-1]
            index_chars_list[1] = index_chars_list[1][:-1]
            index_chars_list[0] += "]"
            index_chars_list[1] += "]"
        print(subline)


            v_blk = numpy.einsum(
                "jbkc,ia->ijkabc",
                eris_OVOV[:,:,k0:k1,c0:c1],
                t1a[:,:],
            )

            v_blk += numpy.einsum(
                "iakc,jb->ijkabc",
                eris_ovOV[:,:,k0:k1,c0:c1],
                t1b[:,:],
            )

            v_blk += numpy.einsum(
                "iakc,jb->ijkabc",
                eris_ovOV[:,:,k0:k1,c0:c1],
                t1b[:,:],
            )

            v_blk += (
                numpy.einsum(
                    "JKBC,ai->iJKaBC",
                    t2bb[:,k0:k1,:,c0:c1],
                    fvo[:,:],
                )
                * 0.5
            )

            v_blk += (
                numpy.einsum(
                    "iKaC,BJ->iJKaBC",
                    t2ab[:,k0:k1,:,c0:c1],
                    fVO[:,:],
                )
                * 2
            )


            w_blk -= (
                numpy.einsum(
                    "ikae,jceb->ijkabc",
  

In [ ]:

    d3 = lib.direct_sum("ia+jb+kc->ijkabc", eIA, eia, eia)
    w = numpy.einsum("jIeA,kceb->IjkAbc", t2ab, numpy.asarray(eris.get_ovvv())) * 2
    w += numpy.einsum("jIbE,kcEA->IjkAbc", t2ab, numpy.asarray(eris.get_ovVV())) * 2
    w += numpy.einsum("jkbe,IAec->IjkAbc", t2aa, numpy.asarray(eris.get_OVvv()))
    w -= numpy.einsum("mIbA,kcjm->IjkAbc", t2ab, numpy.asarray(eris.ovoo)) * 2
    w -= numpy.einsum("jMbA,kcIM->IjkAbc", t2ab, numpy.asarray(eris.ovOO)) * 2
    w -= numpy.einsum("jmbc,IAkm->IjkAbc", t2aa, numpy.asarray(eris.OVoo))
    v = numpy.einsum("jbkc,IA->IjkAbc", numpy.asarray(eris.ovov), t1b)
    v += numpy.einsum("kcIA,jb->IjkAbc", numpy.asarray(eris.ovOV), t1a)
    v += numpy.einsum("kcIA,jb->IjkAbc", numpy.asarray(eris.ovOV), t1a)
    v += numpy.einsum("jkbc,AI->IjkAbc", t2aa, fVO) * 0.5
    v += numpy.einsum("kIcA,bj->IjkAbc", t2ab, fvo) * 2

    rw = r4(w) / d3
    wvd = r4(w * 2 + v) / d3
# 
# 
# 
    d3 = lib.direct_sum('ia+jb+kc->ijkabc', eIA, eia, eia)
    w  = numpy.einsum('jIeA,kceb->IjkAbc', t2ab, numpy.asarray(eris.get_ovvv()).conj()) * 2
    w += numpy.einsum('jIbE,kcEA->IjkAbc', t2ab, numpy.asarray(eris.get_ovVV()).conj()) * 2
    w += numpy.einsum('jkbe,IAec->IjkAbc', t2aa, numpy.asarray(eris.get_OVvv()).conj())
    w -= numpy.einsum('mIbA,kcjm->IjkAbc', t2ab, numpy.asarray(eris.ovoo).conj()) * 2
    w -= numpy.einsum('jMbA,kcIM->IjkAbc', t2ab, numpy.asarray(eris.ovOO).conj()) * 2
    w -= numpy.einsum('jmbc,IAkm->IjkAbc', t2aa, numpy.asarray(eris.OVoo).conj())
    v  = numpy.einsum('jbkc,IA->IjkAbc', numpy.asarray(eris.ovov).conj(), t1b)
    v += numpy.einsum('kcIA,jb->IjkAbc', numpy.asarray(eris.ovOV).conj(), t1a)
    v += numpy.einsum('kcIA,jb->IjkAbc', numpy.asarray(eris.ovOV).conj(), t1a)
    v += numpy.einsum('jkbc,AI->IjkAbc', t2aa, fVO) * .5
    v += numpy.einsum('kIcA,bj->IjkAbc', t2ab, fvo) * 2